# Pipeline de Diarización de Videos
Este notebook sube videos a Google Drive, extrae audio, realiza diarización con pyannote.ai y genera análisis de segmentos silenciosos con detección de escenas.

In [ ]:
# Instalar dependencias necesarias
!apt-get -y update && apt-get -y install ffmpeg
!pip -q install moviepy pydub pyannoteai-sdk==0.4.0 scenedetect==0.6.4 opencv-python==4.9.0.80
!pip -q install google-api-python-client google-auth-httplib2 google-auth-oauthlib

print("✅ Dependencias instaladas correctamente")

In [ ]:
import os
import time
import subprocess
import json
import shutil
import getpass
from pathlib import Path
from google.oauth2.credentials import Credentials
from googleapiclient.discovery import build
from googleapiclient.http import MediaFileUpload
from google.colab import files
from moviepy.video.io.VideoFileClip import VideoFileClip
from pydub import AudioSegment
from pyannoteai.sdk import Client
import cv2
from scenedetect import detect, ContentDetector

print("📦 Librerías importadas")

In [ ]:
# ==========================
# Configuración de credenciales de Google Drive
# ==========================
CLIENT_ID = '448213662107-7fuh61iinmeg78df9epa6ve54qm7mn2o.apps.googleusercontent.com'
CLIENT_SECRET = 'YOUR_GOOGLE_OAUTH_CLIENT_SECRET'
REFRESH_TOKEN = 'YOUR_GOOGLE_OAUTH_REFRESH_TOKEN'

# IDs de carpetas en Drive
FOLDER_VIDEOS = '195TbCi27MxDtzV9RATxQ2bBHwsSDUmzb'  # Carpeta de videos
FOLDER_AUDIOS = '1fjmlPFGe8QEXoPjcAZFMFd6AkErkFVCI'  # Carpeta de audios
FOLDER_RESULTS = '1YkyVktHPFDbKiWADf3KJPnkJPHNz-1uO'  # Carpeta para resultados de diarización

print("🔑 Configuración de Drive cargada")

In [ ]:
# Configurar token de pyannote.ai de forma segura
API_TOKEN = getpass.getpass("🔐 Ingresa tu PYANNOTE.AI API token (no se mostrará): ").strip()
assert API_TOKEN.startswith("sk_"), "❌ El token no parece válido (debe comenzar con 'sk_')"
print("✅ Token de pyannote.ai configurado")

In [ ]:
# ==========================
# Funciones de Google Drive
# ==========================

def _build_drive_service():
    """Construye el servicio de Google Drive con las credenciales configuradas."""
    creds = Credentials(
        None,
        refresh_token=REFRESH_TOKEN,
        client_id=CLIENT_ID,
        client_secret=CLIENT_SECRET,
        token_uri='https://oauth2.googleapis.com/token'
    )
    return build('drive', 'v3', credentials=creds)

def _extract_wav(input_path: str, output_path: str):
    """Extrae audio WAV de un video usando ffmpeg."""
    subprocess.run([
        'ffmpeg', '-y', '-i', input_path,
        '-vn', '-acodec', 'pcm_s16le',
        '-ar', '44100', '-ac', '2',
        output_path
    ], check=True)

def _upload_to_drive(service, local_path: str, drive_name: str, parent_folder_id: str):
    """Sube un archivo a Google Drive en la carpeta especificada."""
    media = MediaFileUpload(local_path)
    metadata = {'name': drive_name, 'parents': [parent_folder_id]}
    file_obj = service.files().create(body=metadata, media_body=media, fields='id,webContentLink').execute()
    # Cerrar file descriptors si existen
    if hasattr(media, '_fd') and getattr(media._fd, 'closed', True) is False:
        media._fd.close()
    return file_obj

def upload_many_to_drive(file_paths, parent_folder_id):
    """Sube múltiples archivos a Drive."""
    service = _build_drive_service()
    uploaded_files = []
    
    for file_path in file_paths:
        if os.path.exists(file_path):
            try:
                file_name = os.path.basename(file_path)
                result = _upload_to_drive(service, file_path, file_name, parent_folder_id)
                uploaded_files.append({
                    'name': file_name,
                    'id': result.get('id'),
                    'url': result.get('webContentLink')
                })
                print(f"📤 Subido: {file_name}")
            except Exception as e:
                print(f"❌ Error subiendo {file_path}: {e}")
        else:
            print(f"⚠️ Archivo no encontrado: {file_path}")
    
    return uploaded_files

print("🔧 Funciones de Drive definidas")

In [ ]:
# ==========================
# Funciones de procesamiento de audio y video
# ==========================

def get_video_stem(video_path):
    """Obtiene el nombre base del video sin extensión, seguro para archivos."""
    base = os.path.basename(video_path)
    stem, _ = os.path.splitext(base)
    # Limpieza ligera para nombres
    safe = "".join(c if c.isalnum() or c in "._- " else "_" for c in stem).strip()
    return safe or "video"

def extract_audio_moviepy(video_path, audio_path):
    """Extrae audio del video usando MoviePy (formato WAV PCM 16-bit)."""
    video = VideoFileClip(video_path)
    video.audio.write_audiofile(audio_path, codec='pcm_s16le')
    video.close()
    print(f"🎵 Audio extraído: {audio_path}")

def diarize_with_sdk(audio_path, api_token):
    """Realiza diarización usando el SDK oficial de pyannote.ai."""
    client = Client(api_token)
    print("📤 Subiendo audio a pyannote.ai...")
    media_url = client.upload(audio_path)
    job_id = client.diarize(media_url)
    print("⏳ Esperando resultado de diarización...")
    
    while True:
        job = client.retrieve(job_id)
        status = job.get("status")
        if status == "succeeded":
            print("✅ ¡Diarización completada!")
            return job["output"]["diarization"]
        elif status in ["failed", "canceled"]:
            raise Exception(f"❌ Job {status}")
        print(f"⏳ Estado: {status}")
        time.sleep(5)

def save_json(data, filename):
    """Guarda datos en formato JSON."""
    with open(filename, "w", encoding='utf-8') as f:
        json.dump(data, f, indent=2, ensure_ascii=False)
    print(f"💾 Guardado: {filename}")

def split_speakers(audio_path, diarization_json, stem_prefix):
    """Separa el audio por hablantes."""
    audio = AudioSegment.from_wav(audio_path)
    speakers = {}
    
    for segment in diarization_json:
        speaker = segment["speaker"]
        start = int(segment["start"] * 1000)
        end = int(segment["end"] * 1000)
        if speaker not in speakers:
            speakers[speaker] = AudioSegment.empty()
        speakers[speaker] += audio[start:end]
    
    out_files = []
    for speaker, seg_audio in speakers.items():
        out_path = f"{stem_prefix}_speaker_{speaker}.wav"
        seg_audio.export(out_path, format="wav")
        out_files.append(out_path)
        print(f"🎤 Audio de {speaker}: {out_path}")
    
    return out_files

print("🎬 Funciones de procesamiento definidas")

In [ ]:
# ==========================
# Funciones de análisis de segmentos silenciosos
# ==========================

def find_silent_segments(diarization_json, audio_duration=None):
    """Encuentra segmentos sin hablantes (silencios)."""
    segments = sorted(diarization_json, key=lambda x: x["start"])
    if not segments:
        return []
    
    silent_segments = []
    segment_id = 1

    # Silencio inicial
    if segments[0]["start"] > 0:
        silent_segments.append({
            "id": segment_id,
            "start": 0.0,
            "end": segments[0]["start"],
            "duration": round(segments[0]["start"] - 0.0, 3)
        })
        segment_id += 1

    # Silencios entre segmentos
    for i in range(len(segments) - 1):
        current_end = segments[i]["end"]
        next_start = segments[i + 1]["start"]
        if next_start > current_end:
            silent_segments.append({
                "id": segment_id,
                "start": current_end,
                "end": next_start,
                "duration": round(next_start - current_end, 3)
            })
            segment_id += 1

    # Silencio final
    if audio_duration and segments[-1]["end"] < audio_duration:
        silent_segments.append({
            "id": segment_id,
            "start": segments[-1]["end"],
            "end": audio_duration,
            "duration": round(audio_duration - segments[-1]["end"], 3)
        })

    return silent_segments

def join_close_segments(silent_segments, video_path, max_gap=2.0):
    """Une segmentos cercanos verificando cambios de escena."""
    print(f"🔗 Uniendo segmentos con gap <= {max_gap} segundos...")
    scenes = detect(video_path, ContentDetector(threshold=27.0))
    scene_times = [(scene[0].get_seconds(), scene[1].get_seconds()) for scene in scenes]

    joined_segments = []
    i = 0
    
    while i < len(silent_segments):
        current_segment = silent_segments[i].copy()
        segments_to_join = [current_segment]
        j = i + 1
        
        while j < len(silent_segments):
            gap = silent_segments[j]["start"] - current_segment["end"]
            if gap <= max_gap:
                segments_to_join.append(silent_segments[j].copy())
                current_segment = silent_segments[j]
                j += 1
            else:
                break

        if len(segments_to_join) == 1:
            joined_segments.append(segments_to_join[0])
            i += 1
        else:
            joined_start = segments_to_join[0]["start"]
            joined_end = segments_to_join[-1]["end"]
            joined_duration = joined_end - joined_start

            # Verificar cambios de escena
            has_scene_change = False
            for scene_start, scene_end in scene_times:
                if (scene_start > joined_start and scene_start < joined_end):
                    has_scene_change = True
                    break

            if has_scene_change:
                print(f"🎬 Cambio de escena detectado en {joined_start:.3f}s - {joined_end:.3f}s")
                joined_segments.extend(segments_to_join)
            else:
                joined_segment = {
                    "id": len(joined_segments) + 1,
                    "start": joined_start,
                    "end": joined_end,
                    "duration": round(joined_duration, 3),
                    "joined_from": [seg["id"] for seg in segments_to_join]
                }
                print(f"🔗 Uniendo {joined_start:.3f}s - {joined_end:.3f}s")
                joined_segments.append(joined_segment)
            i = j

    # Reasignar IDs
    for idx, segment in enumerate(joined_segments, start=1):
        segment["id"] = idx

    return joined_segments

def detect_scene_changes_in_segments(video_path, silent_segments, min_scene_duration=2.0):
    """Detecta cambios de escena dentro de segmentos silenciosos."""
    print("🎬 Detectando cambios de escena...")
    scenes = detect(video_path, ContentDetector(threshold=27.0))
    scene_times = [(scene[0].get_seconds(), scene[1].get_seconds()) for scene in scenes]
    print(f"📊 Detectadas {len(scene_times)} escenas")

    processed_segments = []
    
    for segment in silent_segments:
        segment_start = segment["start"]
        segment_end = segment["end"]
        overlapping_scenes = []
        
        for scene_start, scene_end in scene_times:
            if (scene_start < segment_end and scene_end > segment_start):
                overlap_start = max(scene_start, segment_start)
                overlap_end = min(scene_end, segment_end)
                if overlap_end > overlap_start:
                    overlapping_scenes.append((overlap_start, overlap_end))

        if len(overlapping_scenes) <= 1:
            processed_segments.append(segment)
        else:
            sub_segments = []
            current_id = segment["id"]
            
            for i, (scene_start, scene_end) in enumerate(overlapping_scenes):
                sub_duration = scene_end - scene_start
                if sub_duration >= min_scene_duration:
                    sub_segments.append({
                        "id": f"{current_id}_{i+1}",
                        "start": round(scene_start, 3),
                        "end": round(scene_end, 3),
                        "duration": round(sub_duration, 3),
                        "original_segment_id": current_id,
                        "scene_number": i + 1
                    })
            
            if not sub_segments:
                processed_segments.append(segment)
            else:
                processed_segments.extend(sub_segments)

    # Reasignar IDs
    for idx, segment in enumerate(processed_segments, start=1):
        segment["id"] = idx

    return processed_segments

print("🔍 Funciones de análisis definidas")

In [ ]:
# ==========================
# Función principal de subida y procesamiento
# ==========================

def upload_and_process_video():
    """Sube un video, extrae WAV, sube ambos a Drive y ejecuta el pipeline de diarización."""
    print('📁 Selecciona el archivo de video para subir:')
    uploaded = files.upload()
    if not uploaded:
        print('❌ No se seleccionó ningún archivo')
        return None

    original_name = list(uploaded.keys())[0]
    base = str(Path(original_name).stem)
    wav_name = base + '.wav'

    os.makedirs('temp', exist_ok=True)
    video_path = os.path.join('temp', original_name)
    # Mover el archivo subido a temp conservando el nombre
    shutil.move(original_name, video_path)
    print(f'📹 Video guardado: {video_path}')

    try:
        print('🎵 Extrayendo audio (WAV)...')
        wav_path = os.path.join('temp', wav_name)
        _extract_wav(video_path, wav_path)
        print(f'✅ Audio extraído: {wav_path}')

        print('🔐 Autenticando con Google Drive...')
        service = _build_drive_service()

        print('📤 Subiendo video a Google Drive...')
        video_upload = _upload_to_drive(service, video_path, original_name, FOLDER_VIDEOS)
        print(f"✅ Video subido - ID: {video_upload.get('id')}")

        print('📤 Subiendo audio a Google Drive...')
        audio_upload = _upload_to_drive(service, wav_path, wav_name, FOLDER_AUDIOS)
        print(f"✅ Audio subido - ID: {audio_upload.get('id')}")

        # ===== INICIO DEL PIPELINE DE DIARIZACIÓN =====
        print('\n🚀 === INICIANDO PIPELINE DE DIARIZACIÓN ===')
        
        # Preparar nombres de archivos basados en el video
        video_stem = get_video_stem(video_path)
        audio_file_diarization = f"{video_stem}_audio.wav"
        diarization_json_file = f"{video_stem}_diarization.json"
        silent_joined_file = f"{video_stem}_silent_segments_joined.json"
        silent_with_scenes_file = f"{video_stem}_silent_segments_with_scenes.json"
        
        # Copiar el audio WAV para el procesamiento
        shutil.copy2(wav_path, audio_file_diarization)
        
        # Paso 1: Diarización con pyannote.ai
        print('🎤 Realizando diarización...')
        diarization = diarize_with_sdk(audio_file_diarization, API_TOKEN)
        save_json(diarization, diarization_json_file)
        
        # Paso 2: Separar audios por hablante
        print('👥 Separando audios por hablante...')
        speaker_files = split_speakers(audio_file_diarization, diarization, video_stem)
        
        # Paso 3: Análisis de segmentos silenciosos
        print('🔇 Analizando segmentos silenciosos...')
        audio_duration = AudioSegment.from_wav(audio_file_diarization).duration_seconds
        silent_segments = find_silent_segments(diarization, audio_duration)
        
        # Filtrar silencios > 2s
        long_silent_segments = [s for s in silent_segments if s["duration"] > 2.0]
        for idx, segment in enumerate(long_silent_segments, start=1):
            segment["id"] = idx
        
        print(f"📊 Encontrados {len(long_silent_segments)} segmentos silenciosos > 2s")
        
        # Paso 4: Unir segmentos cercanos
        print('🔗 Uniendo segmentos cercanos...')
        joined_segments = join_close_segments(long_silent_segments, video_path, max_gap=2.0)
        save_json(joined_segments, silent_joined_file)
        print(f"📊 {len(joined_segments)} segmentos después de unir")
        
        # Paso 5: Detectar cambios de escena
        print('🎬 Detectando cambios de escena en segmentos...')
        final_segments = detect_scene_changes_in_segments(video_path, joined_segments, min_scene_duration=2.0)
        save_json(final_segments, silent_with_scenes_file)
        
        # Mostrar resumen de segmentos finales
        print(f"\n📊 === RESUMEN FINAL: {len(final_segments)} segmentos ===")
        for segment in final_segments[:10]:  # Mostrar solo los primeros 10
            if "scene_number" in segment:
                print(f"  🎬 Segmento {segment['id']}: {segment['start']:.3f}s - {segment['end']:.3f}s (Escena {segment['scene_number']})")
            else:
                print(f"  🔇 Segmento {segment['id']}: {segment['start']:.3f}s - {segment['end']:.3f}s")
        
        if len(final_segments) > 10:
            print(f"  ... y {len(final_segments) - 10} segmentos más")
        
        # Paso 6: Subir todos los resultados a Drive
        print('\n📤 Subiendo resultados a Google Drive...')
        files_to_upload = [
            audio_file_diarization,
            diarization_json_file,
            silent_joined_file,
            silent_with_scenes_file
        ] + speaker_files
        
        uploaded_results = upload_many_to_drive(files_to_upload, FOLDER_RESULTS)
        
        # Limpieza de archivos temporales
        print('🧹 Limpiando archivos temporales...')
        time.sleep(0.5)
        temp_files = [video_path, wav_path, audio_file_diarization] + files_to_upload
        for temp_file in temp_files:
            try:
                if os.path.exists(temp_file):
                    os.remove(temp_file)
            except Exception as e:
                print(f"⚠️ No se pudo eliminar {temp_file}: {e}")
        
        # Resultado final
        result = {
            'video': {
                'name': original_name,
                'file_id': video_upload.get('id'),
                'download_url': video_upload.get('webContentLink')
            },
            'audio': {
                'name': wav_name,
                'file_id': audio_upload.get('id'),
                'download_url': audio_upload.get('webContentLink')
            },
            'diarization_results': uploaded_results,
            'segments_summary': {
                'total_segments': len(final_segments),
                'segments_with_scenes': len([s for s in final_segments if 'scene_number' in s]),
                'speakers_found': len(speaker_files)
            }
        }

        print('\n🎉 === PROCESO COMPLETADO EXITOSAMENTE ===')
        print(f"📹 Video: {result['video']['download_url']}")
        print(f"🎵 Audio: {result['audio']['download_url']}")
        print(f"📊 Resultados: {len(uploaded_results)} archivos subidos a Drive")
        print(f"👥 Hablantes encontrados: {result['segments_summary']['speakers_found']}")
        print(f"🎬 Segmentos totales: {result['segments_summary']['total_segments']}")
        
        return result

    except subprocess.CalledProcessError as ffmpeg_error:
        print(f'❌ Error con ffmpeg: {ffmpeg_error}')
        return None
    except Exception as e:
        print(f'❌ Error general: {e}')
        import traceback
        traceback.print_exc()
        return None

print('✅ === SISTEMA LISTO. Ejecuta upload_and_process_video() para iniciar ===')

In [ ]:
# ==========================
# EJECUTAR EL PIPELINE COMPLETO
# ==========================

# Ejecuta esta celda para iniciar todo el proceso:
# 1. Subir video
# 2. Extraer y subir audio
# 3. Realizar diarización
# 4. Generar análisis de segmentos
# 5. Subir todos los resultados a Drive

result = upload_and_process_video()

if result:
    print("\n📋 === RESUMEN FINAL ===")
    print(f"Video procesado: {result['video']['name']}")
    print(f"Hablantes detectados: {result['segments_summary']['speakers_found']}")
    print(f"Segmentos analizados: {result['segments_summary']['total_segments']}")
    print(f"Archivos generados: {len(result['diarization_results'])}")
    print("\n🔗 Enlaces de descarga:")
    print(f"📹 Video: {result['video']['download_url']}")
    print(f"🎵 Audio: {result['audio']['download_url']}")
else:
    print("❌ El procesamiento falló. Revisa los errores anteriores.")

## 📋 Descripción del Pipeline

Este notebook ejecuta un pipeline completo de análisis de video que incluye:

### 🔄 Proceso Automático:
1. **📤 Subida de Video**: Sube el video a Google Drive
2. **🎵 Extracción de Audio**: Extrae audio WAV y lo sube a Drive
3. **🎤 Diarización**: Identifica y separa hablantes usando pyannote.ai
4. **👥 Separación por Hablante**: Genera archivos de audio individuales
5. **🔇 Análisis de Silencios**: Detecta segmentos sin habla
6. **🎬 Detección de Escenas**: Identifica cambios de escena en silencios
7. **📊 Generación de Reportes**: Crea archivos JSON con análisis detallado
8. **☁️ Subida a Drive**: Guarda todos los resultados automáticamente

### 📁 Archivos Generados:
- `{video}_audio.wav` - Audio extraído
- `{video}_diarization.json` - Resultado de diarización
- `{video}_speaker_X.wav` - Audio por cada hablante
- `{video}_silent_segments_joined.json` - Segmentos silenciosos unidos
- `{video}_silent_segments_with_scenes.json` - Análisis final con escenas

### 🔧 Configuración:
- **Umbral de silencio**: 2 segundos mínimo
- **Gap máximo para unir**: 2 segundos
- **Duración mínima de escena**: 2 segundos
- **Umbral de detección de escena**: 27.0

Todos los archivos se nombran automáticamente basándose en el nombre del video original y se organizan en las carpetas correspondientes de Google Drive.